# Memory Allocation and Deallocation, Garbage Collection, and Best Practices

This notebook explains how Python manages memory, how objects are allocated and released, and how garbage collection helps clean up cyclic references. It also shows a few practical habits that keep code easier to reason about and more memory-friendly.

## Learning Goals

- Understand how Python stores objects in memory
- See how reference counting releases unused objects
- Learn why cyclic garbage collection is needed
- Use the `gc` module for inspection and cleanup
- Follow simple best practices for memory-aware Python code

## Python Memory Model

In Python, variables are names that point to objects. The object itself lives somewhere in memory, and Python keeps track of how many references point to that object. When the last reference disappears, the object can usually be released immediately.

For most objects, CPython uses reference counting. It also has a cyclic garbage collector that finds reference cycles that simple counting cannot clean up.

In [1]:
import gc
import sys

## Allocation and Deallocation

Allocation happens when Python creates an object such as an integer, list, string, or dictionary. Deallocation happens when the object is no longer needed and Python can reclaim its memory.

You do not free objects manually in normal Python code. Instead, you remove references and let Python manage the object's lifetime.

In [2]:
data = [1, 2, 3, 4]
alias = data

print('Reference count while two names point to the list:', sys.getrefcount(data))

del alias
print('Reference count after deleting one name:', sys.getrefcount(data))

Reference count while two names point to the list: 3
Reference count after deleting one name: 2


## Garbage Collection and Cycles

Reference counting cannot clean up objects that refer to each other in a cycle. For example, two objects may point to each other even after no external code uses them anymore. The garbage collector can detect and free those unreachable cycles.

The `gc` module lets you inspect and control collection when needed.

In [3]:
class Node:
    def __init__(self, name):
        self.name = name
        self.next = None

first = Node('first')
second = Node('second')
first.next = second
second.next = first

print('Garbage collector enabled:', gc.isenabled())
print('Objects tracked before deleting references:', gc.get_count())

del first
del second
collected = gc.collect()
print('Unreachable objects collected:', collected)

Garbage collector enabled: True
Objects tracked before deleting references: (239, 9, 4)
Unreachable objects collected: 49


## Best Practices

- Prefer clear data ownership so objects are released naturally when no longer needed.
- Avoid keeping long-lived global references to large objects unless they are truly required.
- Break reference cycles in custom classes when they hold external resources such as files, sockets, or database connections.
- Use context managers for resources like files so cleanup happens promptly.
- Profile memory only when necessary. Premature optimization often adds more complexity than value.
- Keep multiprocessing and threading examples inside a main guard on Windows when running scripts.

## Practical Takeaways

Python handles most memory work for you. Your job is usually to write code that does not accidentally hold onto objects longer than needed. When you understand references, cycles, and the garbage collector, it becomes much easier to write predictable and efficient programs.